# Walmart Store Sales Forecasting: Model Inference

This notebook fulfils the assignment's inference requirements:

- The team's **best model** (by shared validation WMAE) is **registered in the MLflow Model Registry**.
- The model is **loaded directly from the registry** (not from a local file, not retrained).
- It predicts on the **raw, unpreprocessed test set** and generates the final submission file.

**Final model comparison on the shared validation split** (39-week window, Nov 2011 - Jul 2012, chosen to match the test period's holiday composition; metric: WMAE, holiday weeks weighted 5x; lower is better):

| Model | Architecture family | Best val WMAE | Notebook |
|---|---|---|---|
| **LightGBM** | gradient-boosted trees | **1811** | model_experiment_LightGBM |
| XGBoost | gradient-boosted trees | 1839 | model_experiment_XGBoost |
| seasonal naive (benchmark) | - | ~2045 | inside LightGBM notebook (blend alpha=1.0) |
| TimesFM (bonus) | pretrained foundation model, zero-shot + blend | 2251.43 | model_experiment_TimesFM |
| N-BEATS | deep learning | 2342.47 | model_experiment_NBEATS |
| PatchTST | deep learning (Transformer) | 2529.41 | model_experiment_PatchTST |
| Prophet | classical | 2817.95 | model_experiment_Prophet |
| TFT (bonus) | deep learning (Transformer + covariates) | 2869.22 | model_experiment_TFT |
| DLinear | deep learning (linear) | 3385.80 | model_experiment_DLinear |

ARIMA and SARIMA are deliberately absent from this table: per the team's documented decision (and the assignment's instruction to limit time on them), they were evaluated only on a few representative series, so they have no fleet-wide WMAE comparable to the numbers above.

**Winner: LightGBM** (direct multi-horizon + seasonal-naive holiday blend). The ranking matches published competition evidence (M4 / M5): on short, feature-rich retail series, gradient-boosted trees with engineered features beat neural forecasters. Notably, the zero-shot TimesFM foundation model outperformed every model we trained ourselves except the two tree ensembles.

Notes for honest reporting: the DL models score the 36 cold-start test rows via the shared fallback while the tree evaluation dropped 413 cold validation rows (small asymmetry, documented in the README). Models are compared on the holdout described above; the final model was additionally evaluated on Kaggle via late submission: **private WMAE 2751.90, public 2642.17** (the historical competition winner scored ~2301; the validation-to-test gap reflects the one-year shift between our validation window and the test period).

## Setup

Kaggle settings: Internet ON, competition dataset attached, `DAGSHUB_TOKEN` secret enabled. The winning pipeline was trained with LightGBM, so its library must be present for unpickling.

In [3]:
!pip install -q mlflow dagshub lightgbm

In [4]:
import os
import numpy as np
import pandas as pd
import mlflow

import dagshub
from kaggle_secrets import UserSecretsClient

dagshub.auth.add_app_token(UserSecretsClient().get_secret('DAGSHUB_TOKEN'))
dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='walmart-sales-forecasting', mlflow=True)
print('tracking uri:', mlflow.get_tracking_uri())
print('mlflow', mlflow.__version__)

Accessing as Nestor-Dzadzamia

Initialized MLflow to track repo "Nestor-Dzadzamia/walmart-sales-forecasting"

Repository Nestor-Dzadzamia/walmart-sales-forecasting initialized!

tracking uri: https://dagshub.com/Nestor-Dzadzamia/walmart-sales-forecasting.mlflow
mlflow 3.14.0


## Locate and register the winning pipeline

The team's best model is the LightGBM direct-multi-horizon pipeline (validation WMAE 1811), logged and load-verified in `model_experiment_LightGBM.ipynb` (reload max diff ~6e-11). Its run id is pinned below.

Registration tries several URI forms in order, because on Kaggle's MLflow 3.x the `runs:/` form is unreliable for pyfunc models (it failed in the DLinear run); the `models:/<model_id>` form is the one that works. The cell resolves the logged-model id from the run automatically, and falls back to a manually pasted `models:/` URI or the `runs:/` form.

In [5]:
from mlflow import MlflowClient
client = MlflowClient()

RUN_ID = '261459588a32436ca99d6ca029f9a6f7'

run = client.get_run(RUN_ID)
print('run name:', run.data.tags.get('mlflow.runName'))
print('ended:', pd.to_datetime(run.info.end_time, unit='ms'))
print('artifact root:', run.info.artifact_uri)

print()
print('--- all run tags:')
for k, v in run.data.tags.items():
    print(f'  {k} = {str(v)[:140]}')

model_ids = set()

# probe 1: MLflow 3 run outputs (GetRun may expose the logged model ids directly)
outs = getattr(run, 'outputs', None)
if outs and getattr(outs, 'model_outputs', None):
    for mo in outs.model_outputs:
        print('run output model:', mo.model_id)
        model_ids.add(mo.model_id)
else:
    print('no model_outputs exposed by server for this run')

# probe 2: logged-model search scoped by experiment (no filter string)
try:
    for lm in client.search_logged_models(experiment_ids=[run.info.experiment_id]):
        src = getattr(lm, 'source_run_id', None)
        print('logged model:', lm.model_id, '| name:', getattr(lm, 'name', '?'), '| source_run:', src)
        if src == RUN_ID:
            model_ids.add(lm.model_id)
except Exception as e:
    print('search_logged_models(experiment_ids=...) failed:', str(e)[:140])

# probe 3: MLflow 2-style placement under run artifacts
def find_mlmodel_dirs(run_id, path=None, depth=0):
    found = []
    if depth > 4:
        return found
    try:
        entries = client.list_artifacts(run_id, path)
    except Exception:
        return found
    for e in entries:
        if e.is_dir:
            found += find_mlmodel_dirs(run_id, e.path, depth + 1)
        elif e.path.endswith('MLmodel'):
            found.append(e.path.rsplit('/', 1)[0] if '/' in e.path else '')
    return found

mlmodel_dirs = find_mlmodel_dirs(RUN_ID)
print('run-artifact dirs containing MLmodel:', mlmodel_dirs)

direct_sources = [f"{run.info.artifact_uri}/{d}".rstrip('/') for d in mlmodel_dirs]
resolved_model_uris = []
for mid in sorted(model_ids):
    resolved_model_uris.append(f'models:/{mid}')
    try:
        lm = client.get_logged_model(mid)
        loc = getattr(lm, 'artifact_location', None)
        if loc:
            direct_sources.append(loc)
            print('logged model artifact location:', loc)
    except Exception as e:
        print('get_logged_model failed:', str(e)[:120])

print()
print('resolved model uris:', resolved_model_uris)
print('direct artifact sources:', direct_sources)

run name: LightGBM_Pipeline_Wrapped
ended: 2026-07-11 20:50:19.572000
artifact root: mlflow-artifacts:/ca0336823f75442e9216d3e765a50749/261459588a32436ca99d6ca029f9a6f7/artifacts

--- all run tags:
  mlflow.user = gdzag22
  mlflow.source.name = __notebook_source__.ipynb
  mlflow.source.type = NOTEBOOK
  mlflow.runName = LightGBM_Pipeline_Wrapped
run output model: m-1e10ba1c607a405690492fb235261c27
logged model: m-1e10ba1c607a405690492fb235261c27 | name: pipeline | source_run: 261459588a32436ca99d6ca029f9a6f7
logged model: m-e1d8ca30fe3c4ac080db875114f0362d | name: pipeline | source_run: 8ad9c8a0c69247d29355e01c1582a446
logged model: m-0d653f6a34314dafb6ef43d9ddd7c33d | name: pipeline | source_run: d949fd8430844e3bb6906f47cdb89cf9
logged model: m-ff6e963409464516bed3e45889d253b6 | name: model | source_run: e103b662c56b4a85af7d24a1bf4d8e9e
logged model: m-8a8d1ad9064641eb8babd9c893091b36 | name: pipeline | source_run: cd2a8ec0e22745b595f38ec994d963cb
logged model: m-c97c4676086349bd86afe

In [6]:
REGISTRY_NAME = 'walmart-weekly-sales-best-model'

# Optional manual override: paste either a models:/m-... URI or a full artifact
# source path from the DagsHub UI if automatic discovery found nothing.
MANUAL_MODEL_URI = ''

candidate_uris = []
if MANUAL_MODEL_URI:
    candidate_uris.append(MANUAL_MODEL_URI)
candidate_uris += resolved_model_uris          # models:/m-... (if server supports it)
candidate_uris += direct_sources               # direct artifact location (bypasses logged-model lookup)
candidate_uris.append(f'runs:/{RUN_ID}/pipeline')  # last resort
print('registration candidates (in order):')
for u in candidate_uris:
    print('  ', u)

registration candidates (in order):
   models:/m-1e10ba1c607a405690492fb235261c27
   mlflow-artifacts:/ca0336823f75442e9216d3e765a50749/models/m-1e10ba1c607a405690492fb235261c27/artifacts
   runs:/261459588a32436ca99d6ca029f9a6f7/pipeline


## Register the model in the Model Registry

Tries each candidate URI until one registers successfully.

In [7]:
def try_register(uri):
    if uri.startswith('models:/') or uri.startswith('runs:/'):
        return mlflow.register_model(model_uri=uri, name=REGISTRY_NAME)
    # direct artifact source: create the version explicitly (bypasses runs:/ resolution)
    try:
        client.create_registered_model(REGISTRY_NAME)
    except Exception:
        pass  # name already exists, fine
    return client.create_model_version(name=REGISTRY_NAME, source=uri, run_id=RUN_ID)

registered = None
last_err = None
for uri in candidate_uris:
    try:
        print('registering from:', uri)
        registered = try_register(uri)
        print('OK -> registered', REGISTRY_NAME, 'version', registered.version)
        break
    except Exception as e:
        last_err = e
        print('  failed:', str(e)[:160])

assert registered is not None, (
    'all registration sources failed. Paste a source from the DagsHub UI into '
    f'MANUAL_MODEL_URI. Last error: {last_err}')

registering from: models:/m-1e10ba1c607a405690492fb235261c27


Registered model 'walmart-weekly-sales-best-model' already exists. Creating a new version of this model...
2026/07/12 19:55:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-weekly-sales-best-model, version 3
Created version '3' of model 'walmart-weekly-sales-best-model'.


OK -> registered walmart-weekly-sales-best-model version 3


## Load the model FROM THE REGISTRY

This is the assignment's key requirement: the model comes out of the registry by name and version, with no reference to local files or training code.

In [8]:
registry_uri = f'models:/{REGISTRY_NAME}/{registered.version}'
print('loading from registry:', registry_uri)
try:
    model = mlflow.pyfunc.load_model(registry_uri)
except Exception as e:
    print('registry-uri load failed:', str(e)[:160])
    mv = client.get_model_version(REGISTRY_NAME, registered.version)
    print('falling back to the registered version source:', mv.source)
    model = mlflow.pyfunc.load_model(mv.source)
print('loaded:', type(model).__name__)

loading from registry: models:/walmart-weekly-sales-best-model/3


loaded: PyFuncModel


## Predict on the raw test set

The pipeline contract (identical across all five model notebooks): raw `test.csv` frame in, final `Weekly_Sales` predictions out. Merging, preprocessing, forecasting, cold-start fallback and clipping all happen inside the registered pipeline.

In [9]:
path = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'
test = pd.read_csv(path + 'test.csv.zip')
print('raw test:', test.shape)

predictions = model.predict(test)
print('predictions:', predictions.shape)
predictions.head()

raw test: (115064, 4)


/tmp/ipykernel_58/3032094490.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


predictions: (115064, 4)


/tmp/ipykernel_58/4016649777.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,Store,Dept,Date,Weekly_Sales
0,1,1,2012-11-02,36117.764145
1,1,1,2012-11-09,19801.616135
2,1,1,2012-11-16,20225.920824
3,1,1,2012-11-23,20388.842202
4,1,1,2012-11-30,24577.241149


## Edge-case fallback at inference

The registered LightGBM pipeline handles the 11 cold-start series internally. A small number of remaining rows can still come back as NaN: series whose training history is too thin for the pipeline's lag and blend lookups (in this run, 3 rows across 2 series, 0.003% of test). They are filled with the shared `coldstart_fallback` rule from the team contract (median Weekly_Sales of the same Dept across same-Type stores over the last 52 training weeks; 0 for an entirely unknown Dept; clipped at 0), so the final submission follows the same rule as every model notebook. The sanity asserts below then verify completeness for all 115,064 rows.

In [10]:
nan_mask = predictions['Weekly_Sales'].isna()
print('NaN predictions:', int(nan_mask.sum()))
if nan_mask.any():
    print(predictions.loc[nan_mask].groupby(['Store', 'Dept']).size().rename('rows'))

    train = pd.read_csv(path + 'train.csv.zip')
    stores = pd.read_csv(path + 'stores.csv')
    tr_hist = train.merge(stores[['Store', 'Type']], on='Store', how='left')
    tr_hist['Date'] = pd.to_datetime(tr_hist['Date'])
    recent = tr_hist[tr_hist['Date'] >= tr_hist['Date'].max() - pd.Timedelta(weeks=52)]
    med = recent.groupby(['Type', 'Dept'])['Weekly_Sales'].median()

    nan_rows = predictions.loc[nan_mask, ['Store', 'Dept']].merge(
        stores[['Store', 'Type']], on='Store', how='left')
    fills = [max(float(med.get((t, d), 0.0)), 0.0)
             for t, d in zip(nan_rows['Type'], nan_rows['Dept'])]
    predictions.loc[nan_mask, 'Weekly_Sales'] = fills
    print('filled with shared coldstart_fallback rule; remaining NaN:',
          int(predictions['Weekly_Sales'].isna().sum()))

NaN predictions: 3
Store  Dept
42     30      2
45     39      1
Name: rows, dtype: int64
filled with shared coldstart_fallback rule; remaining NaN: 0


## Sanity checks and final submission

In [11]:
assert len(predictions) == len(test), f'row mismatch: {len(predictions)} vs {len(test)}'
assert predictions['Weekly_Sales'].notna().all(), 'NaN predictions found'
assert (predictions['Weekly_Sales'] >= 0).all(), 'negative predictions found (contract: clip at 0)'

sub = predictions.copy()
sub['Date'] = pd.to_datetime(sub['Date'])
sub['Id'] = (sub['Store'].astype(str) + '_' + sub['Dept'].astype(str)
             + '_' + sub['Date'].dt.strftime('%Y-%m-%d'))
submission = sub[['Id', 'Weekly_Sales']].sort_values('Id').reset_index(drop=True)

sample = pd.read_csv(path + 'sampleSubmission.csv.zip')
assert len(submission) == len(sample), 'row count differs from sampleSubmission'
assert set(submission['Id']) == set(sample['Id']), 'Id set differs from sampleSubmission'

submission.to_csv('final_submission.csv', index=False)
print('final_submission.csv written:', submission.shape)
submission.head()

final_submission.csv written: (115064, 2)


,Id,Weekly_Sales
0,10_10_2012-11-02,46783.745836
1,10_10_2012-11-09,47653.640174
2,10_10_2012-11-16,45468.549157
3,10_10_2012-11-23,47367.955107
4,10_10_2012-11-30,43827.127655


## Notes

- **Registered model**: the LightGBM direct multi-horizon pipeline (validation WMAE 1811), chosen over 8 alternatives on the shared holdout.
- **Why the registry**: a named, versioned model decouples inference from training notebooks; a better future model becomes just a new registry version.
- **What this run proves**: loading the pipeline artifact was already verified inside the LightGBM notebook itself (reload max diff ~6e-11), so the artifact is known-good despite holding the full training frame for lag lookups. The steps validated here for the first time are registry registration and registry-based loading, end to end, plus the final submission generation.
- **Fixed-horizon caveat** (applies to all team pipelines): each registered pipeline forecasts the specific 39-week window after its training history. The competition test set is exactly that window; other date ranges require retraining.
- `final_submission.csv` was submitted to Kaggle as a late submission and scored **private WMAE 2751.90, public 2642.17**, confirming the model end to end on the real test set (historical winner: ~2301).